(sec:md)=
# Molecular dynamics
Molecular dynamics (MD) simulates the time evolution of a molecular system by numerically integrating Newton's equations of motion. At each timestep, forces acting on every atom are derived from the potential energy function of the underlying force field, and atomic positions and velocities are updated accordingly. By propagating the system over many timesteps, MD generates a trajectory from which thermodynamic quantities (temperature, pressure, free energy) and structural properties (radial distribution functions, conformational populations) can be extracted via statistical mechanics.

VeloxChem provides the `OpenMMDynamics` class as a high-level interface to run MD simulations, built on top of the [OpenMM](https://openmm.org) engine {cite}`vlx_workflow`. Starting from a force field topology prepared with the `MMForceFieldGenerator`, a simulation can be set up and executed in a few lines of Python. 

## Creating a Force-Field

In [1]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("ethanol")

ff_gen = vlx.MMForceFieldGenerator()
ff_gen.create_topology(molecule)

Reading ethanol from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

* Info * Using 6-31G* basis set for RESP charges...                                                                       
* Info * Using GAFF (v2.11) parameters.                                                                                   
         Reference: J. Wang, R. M. Wolf, J. W. Caldwell, P. A. Kollman, D. A. Case, J. Comput. Chem. 2004,
         25, 1157-1174.
                                                                                                                          


(sec:MM-init)=
## Initializing the system

In [2]:
opm_dyn = vlx.OpenMMDynamics()

opm_dyn.create_system_from_molecule(
    molecule,
    ff_gen,
    filename="ethanol",  # system parameters and coordinates written to .xml and .pdb
    residue_name="ETH",
)

* Info * System parameters written to ethanol_system.xml                                                                  
* Info * System coordinates written to ethanol_system.pdb                                                                 


(sec:MD-run)=
## Running a MD simulation

In [3]:
opm_dyn.run_md(
    ensemble="NVT",
    temperature=300,  # in Kelvin
    timestep=2.0,  # in fs
    nsteps=10000,
    snapshots=1000,
    traj_file="ethanol_md.pdb",
)

MD Simulation parameters:
Ensemble: NVT
Temperature: 300 K
Friction: 1.0 1/ps
Timestep: 2.0 fs
Total simulation time in ns: 0.02
Step: 0 / 10000 Time: 0.0 ps
Potential Energy -22.69299030303955 kJ/mol
Kinetic Energy: 16.071847915649414 kJ/mol
Temperature: 184.0951427336841 K
Total Energy: -6.621142387390137 kJ/mol
------------------------------------------------------------
Step: 10 / 10000 Time: 0.02 ps
Potential Energy -20.948712825775146 kJ/mol
Kinetic Energy: 13.188268661499023 kJ/mol
Temperature: 151.06515531949222 K
Total Energy: -7.760444164276123 kJ/mol
------------------------------------------------------------
Step: 20 / 10000 Time: 0.04 ps
Potential Energy -17.902326107025146 kJ/mol
Kinetic Energy: 10.24425220489502 kJ/mol
Temperature: 117.34288936517744 K
Total Energy: -7.658073902130127 kJ/mol
------------------------------------------------------------
Step: 30 / 10000 Time: 0.06 ps
Potential Energy -16.31818199157715 kJ/mol
Kinetic Energy: 11.234346389770508 kJ/mol
Temp

In [4]:
import py3Dmol as p3d

with open("../output_files/ethanol_md.pdb", "r") as f:
    pdb_data = f.read()

view = p3d.view(width=600, height=450)
view.addModelsAsFrames(pdb_data, "pdb", {"keepH": True})
view.setStyle({}, {"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}})
view.animate({"loop": "forward", "reps": 0})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.